## 1. Descrição e Justificativa da Adaptação
Nesta etapa, adaptamos a entrada do modelo para viabilizar o uso de redes pré-treinadas no ImageNet (que exigem 3 canais) com os dados do sensor ASTER (9 bandas).

Utilizamos uma camada de Convolução 1x1 para realizar uma combinação linear das bandas multiespectrais, comprimindo-as em 3 canais latentes antes de entrarem na base da rede. Isso permite que o modelo utilize pesos pré-treinados sem descartar a riqueza de dados das bandas SWIR.

### 2. Código de Construção do Modelo (Foco em Teste)

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, Model

def build_transfer_learning_test_model(input_shape=(128, 128, 9), n_classes=2):
    """
     Esta função constrói um modelo de transferência de aprendizado usando MobileNetV2 como base,
     adaptando a entrada de 9 bandas para 3 canais, e finalizando com uma camada de saída.
     """
    # 1. BASE: MobileNetV2 pré-treinada
    # Instanciamos com 3 canais para garantir o carregamento correto dos pesos [cite: 142]
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(input_shape[0], input_shape[1], 3),
        include_top=False,
        weights='imagenet',
        pooling='avg'
    )
    base_model.trainable = False # Congelado para validação inicial [cite: 65]

    # 2. ENTRADA: Suporta os chips de 9 bandas ASTER [cite: 8, 132]
    inputs = layers.Input(shape=input_shape, name="aster_9_bands_input")

    # 3. ADAPTAÇÃO: Conv 1x1 mapeia 9 canais -> 3 canais latentes
    # Essencial para a evolução do modelo conforme o Artefato A08
    x = layers.Conv2D(3, (1, 1), padding='same', name='spectral_adapter')(inputs)

    # 4. CONEXÃO: Passa o output do adaptador para a base pré-treinada
    x = base_model(x, training=False)

    # 5. SAÍDA: Camada densa final para classificação binária [cite: 24, 40]
    outputs = layers.Dense(n_classes, activation='sigmoid', name='output_tl')(x)

    return Model(inputs, outputs, name="SpectraAI_TL_v1_Fixed")

### 3. Execução do Forward Pass e Documentação de Resultados
Aqui realizamos a verificação das dimensões de entrada/saída para garantir a compatibilidade com os dados do projeto.

In [5]:
# 1. Instanciar o modelo adaptado
# Esta chamada agora deve carregar os pesos sem erros [cite: 41]
model_tl = build_transfer_learning_test_model()

# 2. Criar dado sintético (Simulando 1 chip multiespectral do grupo: 128x128x9)
# Garante a compatibilidade com o output do 'recortar_banda' [cite: 128, 140]
dummy_input = tf.random.uniform((1, 128, 128, 9))

print("--- RELATÓRIO TÉCNICO: ---")
print(f"Dimensão de Entrada (Original ASTER): {dummy_input.shape}")

# 3. Execução do Forward Pass
try:
    # Teste de inferência para verificar fluxo de dados [cite: 120]
    prediction = model_tl(dummy_input, training=False)
    
    print(f"Dimensão de Saída (Predição Final):  {prediction.shape}")
    
    # Validação lógica do sucesso do teste conforme conversa com o grupo
    if prediction.shape == (1, 2):
        print("\nSTATUS: SUCESSO")
        print("DOCUMENTAÇÃO: O modelo processou 9 bandas e retornou probabilidades binárias.")
        print("VERIFICAÇÃO: Conexão entre Adaptador e Base validada com sucesso.")
except Exception as e:
    print(f"\nSTATUS: FALHA NO FORWARD PASS: {e}")

# Exibir o sumário para documentação oficial do Artefato A08 [cite: 46, 115]
model_tl.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
--- RELATÓRIO TÉCNICO: ---
Dimensão de Entrada (Original ASTER): (1, 128, 128, 9)
Dimensão de Saída (Predição Final):  (1, 2)

STATUS: SUCESSO
DOCUMENTAÇÃO: O modelo processou 9 bandas e retornou probabilidades binárias.
VERIFICAÇÃO: Conexão entre Adaptador e Base validada com sucesso.


Model: "SpectraAI_TL_v1_Fixed"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ aster_9_bands_input             │ (None, 128, 128, 9)    │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spectral_adapter (Conv2D)       │ (None, 128, 128, 3)    │            30 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_128            │ (None, 1280)           │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_tl (Dense)               │ (None, 2)              │         2,562 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,260,576 (8.62 MB)

 Trainable params: 2,592 (10.12 KB)

 Non-trainable params: 2,257,984 (8.61 MB)